# Een residentieel energievraagsysteem schatten met Seemingly Unrelated Regression

## Managementsamenvatting

Een regionaal nutsbedrijf schat een residentieel **energievraagsysteem** met twee vergelijkingen — de budgetaandelen van elektriciteit en aardgas als functies van de eigen prijs, de kruisprijs en het huishoudinkomen — met **PROC SYSLIN** volgens de seemingly-unrelated-regression-methode (SUR). De twee aandeelvergelijkingen delen een gemeenschappelijk huishoudbudget, dus hun storingstermen zijn onderling gecorreleerd; SUR schat de vergelijkingen gezamenlijk om die correlatie te benutten, waarbij het tekencoherente eigen- en kruisprijseffecten terugvindt en de kruisvergelijkingscovariantie en restrictietoetsen levert die een vraaganalist nodig heeft.

## Gegevensbronnen

| Dataset | Rijen | Granulariteit | Sleutelvariabelen | Beschrijving |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | één rij per maandelijkse marktobservatie | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Synthetisch maandelijks panel van de residentiële energiemarkt. `lp_elec`/`lp_gas` zijn de log-reële prijzen van elektriciteit en aardgas; `lincome` is het log-reële huishoudinkomen; `w_elec`/`w_gas` zijn de bestedingsbudgetaandelen voor elektriciteit en aardgas, gegenereerd uit een bekende AIDS-achtige vraagstructuur plus gecorreleerde ruis tussen de vergelijkingen. |

# Een residentieel energievraagsysteem schatten met Seemingly Unrelated Regression

Een regionaal gas- en elektriciteitsbedrijf wil begrijpen hoe huishoudens hun energiebudget herverdelen tussen **elektriciteit** en **aardgas** naarmate relatieve prijzen en inkomen veranderen. Het natuurlijke kader is een **vraagsysteem**: een reeks budgetaandeelvergelijkingen die gezamenlijk worden geschat.

Twee kenmerken maken gezamenlijke schatting het juiste instrument:

- De aandeelvergelijkingen putten uit een gemeenschappelijk huishoudbudget, dus hun storingstermen zijn **onderling gecorreleerd** — wanneer een huishouden meer aan elektriciteit uitgeeft, geeft het minder aan gas uit. De vergelijkingen samen schatten met **seemingly unrelated regression (SUR)** benut die correlatie voor efficiëntie.
- De economische theorie legt **lineaire restricties** op (adding-up, homogeniteit, symmetrie) die een systeemschatter rechtstreeks kan afdwingen.

`PROC SYSLIN` met de `SUR`-optie is precies voor deze situatie gemaakt. Het schat elke aandeelvergelijking, schat de kruisvergelijkingscovariantie van de storingstermen, en herschat het systeem met die covariantie om efficiënte, theoriecoherente elasticiteiten te leveren — samen met de kruismodel-covariantiematrix en gezamenlijke restrictietoetsen.

In dit notebook doen we het volgende:

1. Een synthetisch maandelijks energiemarktpanel genereren uit een bekende aandeelstructuur.
2. Een aandeelsysteem met twee vergelijkingen schatten met SUR.
3. De gezamenlijke SUR-fit vergelijken met vergelijking-voor-vergelijking OLS.
4. Een homogeniteitsrestrictie opleggen en de bijbehorende gezamenlijke F-toets aflezen.
5. De coëfficiëntentabel extraheren voor elasticiteitsrapportage.

## Stap 1 — Een synthetisch energiemarktpanel genereren

We simuleren 60 maandelijkse observaties van een klein **energievraagsysteem met twee goederen** (elektriciteit en aardgas). Het gegevensgenererende proces is een gelineariseerd, AIDS-achtig budgetaandeelmodel:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

We bouwen bewust **correlatie tussen de vergelijkingen** in: wanneer huishoudens meer aan elektriciteit uitgeven, geven ze minder aan gas uit, dus `e1` en `e2` zijn negatief gecorreleerd. Een gemeenschappelijke prijsfactor van de energiemarkt stuwt bovendien beide log-prijzen samen. Dit zijn de kenmerken die gezamenlijke SUR-schatting belonen boven het afzonderlijk schatten van elke vergelijking.

In [1]:
GEGEVENS energy;
   CALL streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   DOE month = 1 TOT 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      UITVOER;
   EINDE;

   BEWAREN month lp_elec lp_gas lincome w_elec w_gas;
UITVOEREN;

PROCEDURE GEMIDDELDEN GEGEVENS=energy n mean std MIN MAX maxdec=3;
   VARIABELE w_elec w_gas lp_elec lp_gas lincome;
UITVOEREN;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Stap 2 — Basis-SUR-schatting van het vraagsysteem

We schatten beide aandeelvergelijkingen gezamenlijk. De `SUR`-optie vertelt `PROC SYSLIN` de kruisvergelijkingscovariantie van de storingstermen te schatten en te gebruiken voor een efficiënte feasible-GLS-fit. Twee gelabelde `MODEL`-instructies (`elec` en `gas`) definiëren het systeem; elk regresseert een budgetaandeel op de twee log-prijzen en het log-inkomen.

SYSLIN rapporteert de parameterschattingen van elke vergelijking en, aan het eind, de **Cross-Model Covariance Matrix** — de geschatte covariantie van de storingstermen van de twee vergelijkingen die SUR benut.

In [2]:
PROCEDURE syslin GEGEVENS=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
UITVOEREN;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Stap 3 — Vergelijken met vergelijking-voor-vergelijking OLS

Om te zien wat SUR ons oplevert, herschatten we dezelfde twee vergelijkingen met gewone kleinste kwadraten (de standaardmethode, één vergelijking tegelijk). OLS negeert de correlatie tussen de storingstermen van de vergelijkingen, dus het is consistent maar minder efficiënt dan SUR wanneer die correlatie niet nul is — zoals hier per constructie het geval is.

Het vergelijken van de twee coëfficiëntentabellen laat zien hoe de schattingen en hun standaardfouten verschuiven zodra de systeemstructuur wordt meegenomen.

In [3]:
PROCEDURE syslin GEGEVENS=energy ols;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
UITVOEREN;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Stap 4 — Economische theorie opleggen en toetsen

De vraagtheorie stelt dat nominale prijs-/inkomenseffecten moeten voldoen aan **homogeniteit van graad nul**: het schalen van alle prijzen en het inkomen laat de reële vraag ongewijzigd, dus de prijs- en inkomenscoëfficiënten binnen een vergelijking sommeren tot nul. We leggen dat op aan het elektriciteitsaandeel met een `RESTRICT`-instructie.

SYSLIN herschat het systeem onder de restrictie en rapporteert een F-toets in **Restriction Results** die aangeeft of de restrictie consistent is met de data — een directe, gezamenlijke toets van de homogeniteitshypothese.

In [4]:
PROCEDURE syslin GEGEVENS=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
UITVOEREN;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Stap 5 — Schattingen vastleggen voor elasticiteitsrapportage

Voor een eindrapport bewaren we de geconvergeerde coëfficiënten met `OUTEST=`. De resulterende dataset bevat één rij per vergelijking met de intercept- en hellingsschattingen plus fit-statistieken, die de eigen- en kruisprijselasticiteitsberekeningen van het nutsbedrijf bij de steekproefgemiddelde aandelen voeden. We printen die voor het archief.

In [5]:
PROCEDURE syslin GEGEVENS=energy ON outest=demand_est;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=demand_est noobs;
   TITEL "SUR demand-system coefficient estimates";
UITVOEREN;
TITEL;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## De resultaten interpreteren

**Tekencoherentie.** De SUR-schattingen vinden de substitutiestructuur terug die in de data is ingebouwd. In de elektriciteitsaandeelvergelijking is de **eigen log-prijscoëfficiënt positief** (`LP_ELEC` = 0.112), terwijl de **kruisprijscoëfficiënt negatief** is (`LP_GAS` = -0.098); in de gasaandeelvergelijking spiegelt het patroon dit (eigen `LP_GAS` = 0.114, kruis `LP_ELEC` = -0.075). Elk eigen- en kruisprijseffect is statistisch significant (elke `Pr > |t|` onder 0.005), dus de substitutietekens zijn stevig geïdentificeerd en geen artefacten van een ruizige fit.

**SUR benut de correlatie tussen de vergelijkingen.** De **Cross-Model Covariance Matrix** toont een negatieve off-diagonaal (-0.000068): huishoudens wisselen uitgaven aan elektriciteit uit tegen uitgaven aan gas, precies zoals geconstrueerd. Omdat die covariantie niet nul is, is gezamenlijke SUR-schatting efficiënter dan het afzonderlijk schatten van elke vergelijking — de SUR-standaardfouten in Stap 2 zijn uniform iets kleiner dan de vergelijking-voor-vergelijking OLS-standaardfouten in Stap 3 (de standaardfout van gas `LP_ELEC` daalt bijvoorbeeld van 0.0264 onder OLS naar 0.0255 onder SUR), terwijl de puntschattingen ongewijzigd blijven. Die scherpere inferentie is de opbrengst van het modelleren van het systeem in plaats van twee afzonderlijke regressies.

**De theorie houdt stand.** Het opleggen van **homogeniteit van graad nul** aan het elektriciteitsaandeel (prijs- en inkomenscoëfficiënten die tot nul sommeren) via `RESTRICT` leverde een F-toets in Restriction Results op van 2.14 met `Pr > F` = 0.149. De restrictie wordt **niet verworpen**, dus de homogeniteitsvoorspelling van de consumentenvraagtheorie is consistent met deze steekproef — het nutsbedrijf kan de beperkte, theoriecoherente schattingen met vertrouwen meenemen in de rapportage.

**Operationeel gebruik.** De `OUTEST=`-tabel bewaart één rij per vergelijking met de intercept- en hellingsschattingen en fit-statistieken. Die coëfficiënten laten zich rechtstreeks omzetten in eigen- en kruisprijselasticiteiten en een inkomens- (bestedings-)elasticiteit bij de steekproefgemiddelde aandelen (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 uit Stap 1), wat het nutsbedrijf een verdedigbare, theorieconsistente basis geeft voor vraagvoorspelling en tariefontwerpscenario's.